
# Проект «Работа с данными бизнеса в PySpark»

* Автор: Путилина Елизавета
* Дата: 17.08.2026

В этом проекте вам понадобится поработать с данными сервиса Яндекс Книги, который предоставляет доступ к контенту разных форматов, включая текст, аудио и не только. Руководство сервиса хочет лучше понимать поведение пользователей: какие типы контента они выбирают, как долго его слушают или читают, а также в какие дни недели и через какие платформы (мобильное приложение, веб-версия) это происходит. Эти инсайты позволят улучшить систему рекомендаций и принимать стратегические решения по развитию продукта. Для этого вам понадобится обработать и проанализировать реальные пользовательские данные с помощью PySpark, построить агрегаты и сделать бизнес-выводы на их основе.

Затем вы решите задачи отдела аналитики: нужно проанализировать, как различается суммарное время потребления контента в выходные и будние дни, а также выяснить, для какого типа контента наблюдается такая разница — для взрослого или невзрослого. Дополнительно вас просят преобразовать набор данных и записать его в ClickHouse, чтобы отдел аналитики смог проводить свой анализ на очищенных данных.

### Цели и задачи проекта

**Цель проекта** -  провести анализ поведения пользователей сервиса "Яндекс Книги", чтобы улучшить рекомендации и принимать стратегические решения по развитию продукта.

**Задачи проекта:**

1. Загрузить данные и ознакомиться с ними;
2. Провести трансформацию и преобразование таблиц;
3. Выполнить присоединение таблиц.

### Описание данных

Информация состоит из двух наборов данных (датасетов):

* `bookmate.audition` - содержит данные об активности пользователей;
* `bookmate.content` - содержит данные о контенте сервиса.

Информация из датасета `bookmate.audition`:

* `audition_id` — уникальный идентификатор сессии чтения или прослушивания;
* `puid` — идентификатор пользователя;
* `usage_platform_ru` — название платформы, с помощью которой пользователь взаимодействует с контентом;
* `msk_business_dt_str` — дата и время события (строка, часовой пояс — МСК);
* `app_version` — версия приложения;
* `adult_content_flg` — значение, которое показывает, был ли контент для взрослых (True или False);
* `hours` — длительность сессии чтения или прослушивания в часах;
* `hours_sessions_long` — длительность длинных сессий в часах;
* `kids_content_flg` — значение, которое показывает, был ли это детский контент (True или False);
* `main_content_id` — идентификатор основного контента;
* `usage_geo_id` — идентификатор географического местоположения пользователя.

Информация из датасета `bookmate.content`:

* `main_content_id` — идентификатор основного контента;
* `main_author_id` — идентификатор основного автора контента;
* `main_content_type` — тип контента: аудио, текст или другой;
* `main_content_name` — название контента;
* `main_content_duration_hours` — длительность контента в часах;
* `published_topic_title_list` — список жанров или тем контента.

### Содержание проекта

1. [Загрузка данных и знакомство с ними](#chapter1)
2. [Трансформация и преобразование таблиц](#chapter2)
3. [Соединение таблиц](#chapter3)

## 1. Загрузка данных и знакомство с ними<a class="anchor" id="chapter1"></a>

* Загрузим необходимые библиотеки для анализа данных.

In [1]:
# Импортируем библиотеки
import pyspark
import csv
import io
from pyspark.sql import SQLContext, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, dayofweek, when
from pyspark.sql.types import IntegerType

In [2]:
# Создаём объект SparkSession, добавляем конфигурации для подключения
spark = SparkSession.builder\
  .master("local[*]")\
  .appName("projectTest")\
  .getOrCreate()

# Считываем CSV-файлы и сохраняем в датафрейм
try:
    content_df = spark.read.csv('./content.csv', inferSchema = True)
    audition_df = spark.read.csv('./audition.csv', inferSchema = True)
    print('=' * 50)
    print('Данные успешно загружены.')
    print('=' * 50)
except Exception as e:
    print('=' * 50)
    print(f'Произошла ошибка при загрузке данных: {e}.')
    print('=' * 50)

Данные успешно загружены.


In [3]:
# Переименовываем столбцы датасета `audition_df`
audition_df = audition_df.select(
    col("_c0").alias("usage_geo_id"),
    col("_c1").alias("audition_id"),
    col("_c2").alias("puid"),
    col("_c3").alias("usage_platform_ru"),
    col("_c4").alias("msk_business_dt_str"),
    col("_c5").alias("app_version"),
    col("_c6").alias("adult_content_flg"),
    col("_c7").cast("integer").alias("hours"),
    col("_c8").alias("hours_sessions_long"),
    col("_c9").alias("kids_content_flg"),
    col("_c10").alias("main_content_id"),
    col("_c11").alias("usage_geo_id_name"),
    col("_c12").alias("usage_country_name")
)

# Переименовываем столбцы датасета `content_df`
content_df = content_df.select(
    col("_c0").alias("main_content_id"),
    col("_c1").alias("main_content_type"),
    col("_c2").alias("main_content_name"),
    col("_c3").alias("main_content_duration_hours"),
    col("_c4").alias("published_topic_title_list"),
    col("_c5").alias("main_author_name")
)

# Выполняем действие для применения трансформаций
content_df.show()
audition_df.show()

+---------------+-----------------+--------------------+---------------------------+--------------------------+--------------------+
|main_content_id|main_content_type|   main_content_name|main_content_duration_hours|published_topic_title_list|    main_author_name|
+---------------+-----------------+--------------------+---------------------------+--------------------------+--------------------+
|       A00hxVIL|             Book|Давай поговорим о...|                      4.151|      'Синхронизировано...|        Карл Ричардс|
|       A03if3j5|        Comicbook|Минимализм из ком...|                  2.6854546|      'Уборка и организ...|Элизабет Энрайт Ф...|
|       A05XPJgr|             Book| Гоните ваши денежки|                  6.9114366|      'Художественная л...|Наталья Александрова|
|       A06fBP22|             Book|Крайон. Судьбу мо...|                  3.5218182|      'Эзотерика', 'Окк...|        Тамара Шмидт|
|       A0FkgwIl|        Comicbook|Майор Гром. Допол...|             

+------------+-----------+--------------------+-----------------+-------------------+-----------+-----------------+-----+-------------------+----------------+---------------+--------------------+------------------+
|usage_geo_id|audition_id|                puid|usage_platform_ru|msk_business_dt_str|app_version|adult_content_flg|hours|hours_sessions_long|kids_content_flg|main_content_id|   usage_geo_id_name|usage_country_name|
+------------+-----------+--------------------+-----------------+-------------------+-----------+-----------------+-----+-------------------+----------------+---------------+--------------------+------------------+
|         162|          0|68296628-f9d6-11e...|          Станция|         2024-11-26|       null|            false|    0| 0.0377777777777777|            true|       oCURrBKV|              Алматы|         Казахстан|
|         213|          1|682966dc-f9d6-11e...|          Станция|         2024-11-26|       null|            false|    0|                0.0

* Выведем первые 10 строк датасета `audition_df`.

In [4]:
audition_df.toPandas().head(10)

,usage_geo_id,audition_id,puid,usage_platform_ru,msk_business_dt_str,app_version,adult_content_flg,hours,hours_sessions_long,kids_content_flg,main_content_id,usage_geo_id_name,usage_country_name
0,162,0,68296628-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,False,0,0.037778,True,oCURrBKV,Алматы,Казахстан
1,213,1,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,False,0,0.000000,True,qOL0JJL5,Москва,Россия
2,63,2,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,False,0,0.000000,True,ndM5nzgT,Иркутск,Россия
3,2,4,68296704-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,True,0,0.000000,False,HW5y0HqU,Санкт-Петербург,Россия
4,28,6,68296722-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,False,0,0.000000,True,DDP9Jenb,Махачкала,Россия
5,38,7,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,Музыка iOS,2024-11-26,697.165724,True,0,0.489079,False,R7tyI6r6,Волгоград,Россия
6,11162,8,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,Музыка iOS,2024-11-26,698.166333,True,0,0.184727,False,R7tyI6r6,Свердловская область,Россия
7,11119,9,6829675e-f9d6-11ef-be00-c2c9fa6fd3d5,Букмейт Android,2024-11-26,6.7,True,1,1.453056,False,QBFiEtWi,Республика Татарстан,Россия
8,47,10,6829677c-f9d6-11ef-be00-c2c9fa6fd3d5,Букмейт Android,2024-11-26,6.5,True,0,0.539982,False,xnwMA9a7,Нижний Новгород,Россия
9,194,13,6829679a-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,True,0,0.000000,True,RrDJWAfV,Саратов,Россия


* Выведем первые 10 строк датасета `content_df`.

In [5]:
content_df.toPandas().head(10)

,main_content_id,main_content_type,main_content_name,main_content_duration_hours,published_topic_title_list,main_author_name
0,A00hxVIL,Book,Давай поговорим о твоих доходах и расходах,4.151000,"'Синхронизировано', 'Бизнес', 'Саморазвитие', ...",Карл Ричардс
1,A03if3j5,Comicbook,Минимализм из комнаты в комнату. Пошаговая сис...,2.685455,"'Уборка и организация пространства', 'Домашние...",Элизабет Энрайт Филлипс
2,A05XPJgr,Book,Гоните ваши денежки,6.911437,"'Художественная литература', 'Детективы'",Наталья Александрова
3,A06fBP22,Book,Крайон. Судьбу можно изменить! Как воплотить в...,3.521818,"'Эзотерика', 'Оккультизм'",Тамара Шмидт
4,A0FkgwIl,Comicbook,Майор Гром. Дополнительные материалы к тому №6,0.203636,'Комиксы',Артём Габрелянов
5,A0OKAjnf,Audiobook,Радикальное Прощение: родители и дети. Почему ...,5.078055,"'Психология', 'Саморазвитие', 'Аудио'",Колин Типпинг
6,A0e7vsJT,Book,Обоняние. Увлекательное погружение в науку о з...,8.896182,"'Наука', 'Психология'",Паоло Пелоси
7,A0fAe01q,Audiobook,Амулет ведьмы,10.473333,"'Художественная литература', 'Фэнтези', 'Темно...",Анна Безбрежная
8,A0mSkD2g,Audiobook,Правила инвестирования Уоррена Баффетта,9.809444,"'Биографии и мемуары', 'Бизнес', 'Инвестиции',...",Джереми Миллер
9,A0oeFBMU,Audiobook,Закон трех отрицаний,14.834167,"'Художественная литература', 'Детективы', 'Ауд...",Александра Маринина


* Выведем схемы датасетов `audition_df` и `content_df`.

In [6]:
# Просмотр схем данных
print('Схема данных `audition_df`')
audition_df.printSchema()

print('Схема данных `content_df`')
content_df.printSchema()

Схема данных `audition_df`
root
 |-- usage_geo_id: integer (nullable = true)
 |-- audition_id: integer (nullable = true)
 |-- puid: string (nullable = true)
 |-- usage_platform_ru: string (nullable = true)
 |-- msk_business_dt_str: string (nullable = true)
 |-- app_version: string (nullable = true)
 |-- adult_content_flg: boolean (nullable = true)
 |-- hours: integer (nullable = true)
 |-- hours_sessions_long: double (nullable = true)
 |-- kids_content_flg: boolean (nullable = true)
 |-- main_content_id: string (nullable = true)
 |-- usage_geo_id_name: string (nullable = true)
 |-- usage_country_name: string (nullable = true)

Схема данных `content_df`
root
 |-- main_content_id: string (nullable = true)
 |-- main_content_type: string (nullable = true)
 |-- main_content_name: string (nullable = true)
 |-- main_content_duration_hours: double (nullable = true)
 |-- published_topic_title_list: string (nullable = true)
 |-- main_author_name: string (nullable = true)



* Выведем объём каждого датасета.

In [7]:
# Просмотр числа строк в каждом датасете

print('Объём данных в датасете об активности пользователей `audition_df`:')
print(f'Количество строк: {audition_df.count()};')
print(f'Количество столбцов: {len(audition_df.columns)}.')
print('\n')

print('Объём данных в датасете с данными контента `content_df`:')
print(f'Количество строк: {content_df.count()};')
print(f'Количество столбцов: {len(content_df.columns)}.')

Объём данных в датасете об активности пользователей `audition_df`:


Количество строк: 1002896;
Количество столбцов: 13.


Объём данных в датасете с данными контента `content_df`:
Количество строк: 31668;
Количество столбцов: 6.


**ВЫВОДЫ ПО РАЗДЕЛУ 1**

Датасет `audition_df` содержит 1002896 строк и 13 столбцов, в которых представлена информация об активности пользователей.

Изучим типы данных и их корректность:

1. **Строковые данные (string).** В датасете представлено 7 столбцов с данными такого типа.
* `puid`, `usage_platform_ru`, `app_version`, `main_content_id`, `usage_geo_id_name`, `usage_country_name` содержат строковую информацию. Тип данных `string` корректен;
* `msk_business_dt_str` содержит дату и время события (строка, часовой пояс — МСК). Тип данных необходимо привести к типу `timestamp`;

2. **Целочисленные значения (integer).** В датасете представлено 3 столбца с данными такого типа.
* `usage_geo_id`, `audition_id`, `hours` содержат целочисленные значения. Тип данных `integer` корректен;

3. **Логический тип (boolean).** В датасете представлено 2 столбца с данными такого типа.
* `adult_content_flg`, `kids_content_flg` содержат метки (True/False). Тип данных `boolean` корректен;

4. **Число с плавающей точкой (двойная точность - double).** В датасете представлен 1 столбец с данными такого типа.
* `hours_sessions_long` — длительность длинных сессий в часах. Тип данных `double` корректен.

В ходе анализа датасета `audition_df` было выявлено, что в 1 из 13 столбцов (`msk_business_dt_str`) тип данных представлен некорректно. 

Названия столбцов представлены в едином стиле "snake_case".


Датасет `content_df` содержит 31668 строк и 6 столбцов, в которых представлена информация о контенте сервиса.

Изучим типы данных и их корректность:

1. **Строковые данные (string).** В датасете представлено 5 столбцов с данными такого типа.
* `main_content_id`, `main_content_type`, `main_content_name`, `published_topic_title_list`, `main_author_name` содержат строковую информацию. Тип данных `string` корректен;

2. **Число с плавающей точкой (двойная точность - double).** В датасете представлен 1 столбец с данными такого типа.
* `main_content_duration_hours` — длительность контента в часах. Тип данных `double` корректен.

В ходе анализа датасета `content_df` было выявлено, что все типы данных столбцов представлены корректно.

Названия столбцов представлены в едином стиле "snake_case".

## 2. Трансформация и преобразование таблиц<a class="anchor" id="chapter2"></a>

* Из датасета `audition_df` выберем столбцы `puid` и `hours_sessions_long`. Создадим новый столбец `minutes_sessions_long`, в котором значение `hours_sessions_long` будет умножено на 60 для расчётов в минутах. Заменим тип данных этого столбца на `int`;
* Выведем первые десять строк полученной таблицы.

In [8]:
cast_audition_df = audition_df.select(
    F.col('puid'), 
    F.col('hours_sessions_long'),
    (F.col('hours_sessions_long') * 60).cast(IntegerType()).alias('minutes_sessions_long')
)

# Выводим первые 10 строк
cast_audition_df.toPandas().head(10)

,puid,hours_sessions_long,minutes_sessions_long
0,68296628-f9d6-11ef-be00-c2c9fa6fd3d5,0.037778,2
1,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,0.000000,0
2,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,0.000000,0
3,68296704-f9d6-11ef-be00-c2c9fa6fd3d5,0.000000,0
4,68296722-f9d6-11ef-be00-c2c9fa6fd3d5,0.000000,0
5,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,0.489079,29
6,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,0.184727,11
7,6829675e-f9d6-11ef-be00-c2c9fa6fd3d5,1.453056,87
8,6829677c-f9d6-11ef-be00-c2c9fa6fd3d5,0.539982,32
9,6829679a-f9d6-11ef-be00-c2c9fa6fd3d5,0.000000,0


* К предыдущему запросу добавим новый столбец `is_weekend`, который покажет, был ли этот день рабочим (False — рабочий, True — выходной); 
* Сразу добавим столбец `adult_content_flg`;
* Выведем первые десять строк полученной таблицы.

In [23]:
# Добавление нового столбца is_weekend (1 - воскресенье, 7 - суббота) - покажет True, если выходной
cast_audition_df = (audition_df.withColumn('is_weekend', 
                                        F.dayofweek(F.to_date(F.col('msk_business_dt_str'), "yyyy-MM-dd")).isin([1, 7]))
                              ).select(
                                       F.col('puid'),
                                       F.col('adult_content_flg'),
                                       F.col('hours_sessions_long'),
                                       (F.col('hours_sessions_long') * 60).cast(IntegerType()).alias('minutes_sessions_long'),
                                       F.col('is_weekend')
)

# Выводим первые 10 строк
cast_audition_df.toPandas().head(10)

,puid,adult_content_flg,hours_sessions_long,minutes_sessions_long,is_weekend
0,68296628-f9d6-11ef-be00-c2c9fa6fd3d5,False,0.037778,2,False
1,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,False,0.000000,0,False
2,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,False,0.000000,0,False
3,68296704-f9d6-11ef-be00-c2c9fa6fd3d5,True,0.000000,0,False
4,68296722-f9d6-11ef-be00-c2c9fa6fd3d5,False,0.000000,0,False
5,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,True,0.489079,29,False
6,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,True,0.184727,11,False
7,6829675e-f9d6-11ef-be00-c2c9fa6fd3d5,True,1.453056,87,False
8,6829677c-f9d6-11ef-be00-c2c9fa6fd3d5,True,0.539982,32,False
9,6829679a-f9d6-11ef-be00-c2c9fa6fd3d5,True,0.000000,0,False


* Найдём количество строк с дубликатами в итоговом датасете.

In [29]:
# 1. Считаем общее количество строк
total_count = cast_audition_df.count()
print(f'Всего строк: {total_count}')

# 2. Считаем количество уникальных строк
uniq_count = cast_audition_df.distinct().count()
print(f'Уникальных строк: {uniq_count}')

# 3. Считаем количество строк с дубликатами
duplicate_count = total_count - uniq_count
print(f'Строк с дубликатами: {duplicate_count}')

Всего строк: 1002896


Уникальных строк: 731603
Строк с дубликатами: 271293


* Избавимся от пропусков в итоговой таблице.

In [27]:
clean_df = cast_audition_df.distinct()

print(f'Стало строк: {clean_df.count()}')

Стало строк: 731603


* Рассчитаем суммы значений `minutes_sessions_long` для выходных и будних дней. Назовём этот столбец `total_minutes`. Отсортируем по `is_weekend` и сравним значения.

In [30]:
# Аггрегация значений общего числа минут в рабочие (False) и выходные дни (True)
agg_df_is_week = (clean_df.groupBy('is_weekend')
                          .agg(F.sum('minutes_sessions_long').alias('total_minutes'))
                          .orderBy(F.col('is_weekend').asc()))

agg_df_is_week.show()

+----------+-------------+
|is_weekend|total_minutes|
+----------+-------------+
|     false|     17217123|
|      true|      6419004|
+----------+-------------+



* Проделаем аналогичный анализ отдельно для взрослого и невзрослого контента на основе столбца `adult_content_flg`.  В результате должен получиться датасет с четырьмя строками;

* Отсортируем датасет сначала по возрастному рейтингу, а затем по выходным.

In [31]:
# Проводим аналогичный анализ для взрослого контента
agg_df_adult = (clean_df.groupBy('is_weekend', 'adult_content_flg')
                                .agg(F.sum('minutes_sessions_long').alias('total_minutes'))
                                .orderBy(['adult_content_flg','is_weekend'], ascending = True))

agg_df_adult.show()

+----------+-----------------+-------------+
|is_weekend|adult_content_flg|total_minutes|
+----------+-----------------+-------------+
|     false|            false|      2674414|
|      true|            false|      1240710|
|     false|             true|     14542709|
|      true|             true|      5178294|
+----------+-----------------+-------------+



**ВЫВОДЫ ПО РАЗДЕЛУ 2**

1. В результате проделанной работы, был создан датасет со столбцами:
* `puid` - идентификатор пользователя;
* `adult_content_flg` - метка, которая показывает, был ли контент для взрослых (True/False);
* `hours_sessions_long` - длительность длинных сессий в часах;
* `minutes_sessions_long` - длительность длинных сессий в минутах;
* `is_weekend` - метка, которая показывает, был ли этот день рабочим (False — рабочий, True — выходной).

2. Проведён анализ на наличие дубликатов.

*  Всего строк в датасете: 1002896;
* Уникальных строк: 731603;
* Строк с дубликатами: 271293.

 Строки с дубликатами были удалены. Всего строк в итоговом датасете: 731603.

3. Рассчитана сумма значений `minutes_sessions_long` для выходных и будних дней.

| День недели | Сумма длинных сессий, мин. | Доля длинных сессий |
| :--- | :--- | :--- |
| Будние дни | 17 217 123 | 72,8 % |
| Выходные | 6 419 004 | 27,2 % |

В будние дни пользователи дольше проводят времени в приложениях сервиса (~ 73%), по сравнению с выходными (~ 27%).

4. Рассчитана сумма значений `minutes_sessions_long` для взрослого и невзрослого контента в выходные/будние дни.


| День недели | Тип контента | Сумма длинных сессий, мин. |
| :--- | :--- | :--- |
| Будние дни | Для детей | 2 674 414 |
| Выходные | Для детей | 1 240 710 |
| Будние дни | Для взрослых | 14 542 709 |
| Выходные | Для взрослых | 5 178 294 |


**Ключевые инсайты**

* **Доминирование сегмента «Взрослый контент».** Контент для взрослых пользователей формирует подавляющую часть активности на платформе: 73.6% в будни и 26.2% в выходные. Суммарно на этот сегмент приходится ~80% всего времени потребления. Это указывает на то, что ядро аудитории сервиса составляют взрослые пользователи;

* **Выраженная сезонность по дням недели.** Наблюдается значительное снижение общего времени потребления в выходные дни. Коэффициент падения составляет примерно 2.8 раза. Данная тенденция прослеживается синхронно для обоих типов контента, что позволяет исключить гипотезу о смещении предпочтений аудитории в выходные в пользу детского контента;

* Стабильность структуры потребления. Соотношение долей между взрослым и детским контентом остаётся практически неизменным:
    * В будни: доля взрослого контента ≈ 84,5%;
    * В выходные: доля взрослого контента ≈ 80,7%.

Незначительное изменение пропорций свидетельствует о консистентности пользовательских привычек вне зависимости от дня недели.

Выявленные инсайты могут быть обусловлены следующими поведенческими факторами: 

* **Фоновое потребление в будни.** Высокие показатели в будние дни, особенно для взрослого контента, может говорить о потреблении контента в транспорте или во время рутинных задач (например, по дороге на работу/учёбу). Большое количество относительно коротких сессий накапливается в значительный суммарный объём;

* **Изменение распорядка в выходные.** Снижение активности в выходные может быть связано с перераспределением свободного времени на офлайн‑активности, семейные дела или другие виды досуга, не предполагающие длительного использования сервиса;

* **Специфика детского контента.** Стабильно низкая доля детского контента может указывать на особенности использования сервиса родителями (например, включение контента на ограниченное время), либо на меньшую вовлечённость этой категории пользователей в «длинные сессии».

## 3. Соединение таблиц<a class="anchor" id="chapter3"></a>

* Объединим таблицы `audition_df` и `content_df` по столбцу `main_content_id`;
* Посмотрим, сколько строк получилось после объединения. Убедимся, что количество строк соответствует ожиданиям.

In [32]:
# Объединение по столбцу main_content_id
joined_df = audition_df.join(content_df, on = 'main_content_id', how = 'left')

# Выведем первые 10 строк
joined_df.toPandas().head(10)

,main_content_id,usage_geo_id,audition_id,puid,usage_platform_ru,msk_business_dt_str,app_version,adult_content_flg,hours,hours_sessions_long,kids_content_flg,usage_geo_id_name,usage_country_name,main_content_type,main_content_name,main_content_duration_hours,published_topic_title_list,main_author_name
0,oCURrBKV,162,0,68296628-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,False,0,0.037778,True,Алматы,Казахстан,Audiobook,Фиксики. Интернет. Доступ разрешен,0.830278,"'Детская проза и поэзия', 'Аудио'",Александр Шаронов
1,qOL0JJL5,213,1,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,False,0,0.000000,True,Москва,Россия,Audiobook,Собачка Соня на даче,1.276111,"'Детская проза и поэзия', 'Аудио'",Андрей Усачев
2,ndM5nzgT,63,2,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,False,0,0.000000,True,Иркутск,Россия,Audiobook,Мышонок Тим. А что вы мне подарите?,0.127500,"'Сказки и фольклор', 'Детская проза и поэзия',...",Анна Казалис
3,HW5y0HqU,2,4,68296704-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,True,0,0.000000,False,Санкт-Петербург,Россия,Audiobook,Белые ночи,2.007500,"'Художественная литература', 'Русская литерату...",Фёдор Достоевский
4,DDP9Jenb,28,6,68296722-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,False,0,0.000000,True,Махачкала,Россия,Audiobook,Дикие лебеди,0.917500,"'Детская проза и поэзия', 'Аудио'",Ганс Христиан Андерсен
5,R7tyI6r6,38,7,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,Музыка iOS,2024-11-26,697.165724,True,0,0.489079,False,Волгоград,Россия,Audiobook,Завет воды,30.842222,"'Художественная литература', 'Семейные саги', ...",Абрахам Вергезе
6,R7tyI6r6,11162,8,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,Музыка iOS,2024-11-26,698.166333,True,0,0.184727,False,Свердловская область,Россия,Audiobook,Завет воды,30.842222,"'Художественная литература', 'Семейные саги', ...",Абрахам Вергезе
7,QBFiEtWi,11119,9,6829675e-f9d6-11ef-be00-c2c9fa6fd3d5,Букмейт Android,2024-11-26,6.7,True,1,1.453056,False,Республика Татарстан,Россия,Audiobook,Файролл. Книга 10. Два огня,20.400833,"'Художественная литература', 'ЛитРПГ', 'Фэнтез...",Андрей Васильев
8,xnwMA9a7,47,10,6829677c-f9d6-11ef-be00-c2c9fa6fd3d5,Букмейт Android,2024-11-26,6.5,True,0,0.539982,False,Нижний Новгород,Россия,Book,Откуда берутся дети? Краткий путеводитель по п...,10.504127,"'Здоровье', 'Беременность и роды', 'Медицина'",Ася Казанцева
9,RrDJWAfV,194,13,6829679a-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,None,True,0,0.000000,True,Саратов,Россия,Audiobook,Лунтик. Возвращение домой,1.104445,"'По мотивам', 'Детская проза и поэзия', 'Аудио'",Екатерина Вечеркова


In [34]:
# Подсчет количества строк после объединения
print(f'Количество строк после объединения: {joined_df.count()}')
print(f'Количество столбцов после объединения: {len(joined_df.columns)}')

Количество строк после объединения: 1002896
Количество столбцов после объединения: 18


Было применено "левое" присоединение датасета `content_df` к датасету `audition_df`, потому что мы хотим сохранить все сессии из `audition_df`, даже если для какой-то сессии вдруг не найдётся совпадения в таблице контента (например, контент удалили, или ID потерялся). В таком случае поля из `content_df` будут `null`, но сама сессия не пропадёт. 

Таким образом, количество строк после объединения совпадает с количеством строк датасета `audition_df` (1002896).

* Удалим все лишние столбцы из объединённой таблицы, которые не нужны для дальнейшего анализа: `main_author_id`, `app_version`, `usage_geo_id`;
* Выведем первые десять строк.

In [40]:
# Удаление столбцов из объединённой таблицы
new_df = joined_df.drop('main_author_id', 'app_version', 'usage_geo_id')

# Выведем первые десять строк
new_df.toPandas().head(10)

,main_content_id,audition_id,puid,usage_platform_ru,msk_business_dt_str,adult_content_flg,hours,hours_sessions_long,kids_content_flg,usage_geo_id_name,usage_country_name,main_content_type,main_content_name,main_content_duration_hours,published_topic_title_list,main_author_name
0,oCURrBKV,0,68296628-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,False,0,0.037778,True,Алматы,Казахстан,Audiobook,Фиксики. Интернет. Доступ разрешен,0.830278,"'Детская проза и поэзия', 'Аудио'",Александр Шаронов
1,qOL0JJL5,1,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,False,0,0.000000,True,Москва,Россия,Audiobook,Собачка Соня на даче,1.276111,"'Детская проза и поэзия', 'Аудио'",Андрей Усачев
2,ndM5nzgT,2,682966dc-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,False,0,0.000000,True,Иркутск,Россия,Audiobook,Мышонок Тим. А что вы мне подарите?,0.127500,"'Сказки и фольклор', 'Детская проза и поэзия',...",Анна Казалис
3,HW5y0HqU,4,68296704-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,True,0,0.000000,False,Санкт-Петербург,Россия,Audiobook,Белые ночи,2.007500,"'Художественная литература', 'Русская литерату...",Фёдор Достоевский
4,DDP9Jenb,6,68296722-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,False,0,0.000000,True,Махачкала,Россия,Audiobook,Дикие лебеди,0.917500,"'Детская проза и поэзия', 'Аудио'",Ганс Христиан Андерсен
5,R7tyI6r6,7,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,Музыка iOS,2024-11-26,True,0,0.489079,False,Волгоград,Россия,Audiobook,Завет воды,30.842222,"'Художественная литература', 'Семейные саги', ...",Абрахам Вергезе
6,R7tyI6r6,8,68296740-f9d6-11ef-be00-c2c9fa6fd3d5,Музыка iOS,2024-11-26,True,0,0.184727,False,Свердловская область,Россия,Audiobook,Завет воды,30.842222,"'Художественная литература', 'Семейные саги', ...",Абрахам Вергезе
7,QBFiEtWi,9,6829675e-f9d6-11ef-be00-c2c9fa6fd3d5,Букмейт Android,2024-11-26,True,1,1.453056,False,Республика Татарстан,Россия,Audiobook,Файролл. Книга 10. Два огня,20.400833,"'Художественная литература', 'ЛитРПГ', 'Фэнтез...",Андрей Васильев
8,xnwMA9a7,10,6829677c-f9d6-11ef-be00-c2c9fa6fd3d5,Букмейт Android,2024-11-26,True,0,0.539982,False,Нижний Новгород,Россия,Book,Откуда берутся дети? Краткий путеводитель по п...,10.504127,"'Здоровье', 'Беременность и роды', 'Медицина'",Ася Казанцева
9,RrDJWAfV,13,6829679a-f9d6-11ef-be00-c2c9fa6fd3d5,Станция,2024-11-26,True,0,0.000000,True,Саратов,Россия,Audiobook,Лунтик. Возвращение домой,1.104445,"'По мотивам', 'Детская проза и поэзия', 'Аудио'",Екатерина Вечеркова


In [41]:
# Количество столбцов после удаления
print(f'Количество столбцов после удаления: {len(new_df.columns)}')

Количество столбцов после удаления: 16


* Посчитаем количество уникальных пользователей `puid` в объединённой таблице. Сравним это с количеством пользователей в изначальной таблице `audition_df`. Объясним разницу, если она есть.

In [45]:
# Уникальные пользователи в объединённой таблице
joined_user_count = new_df.distinct().count()

In [46]:
# Уникальные пользователи в исходной таблице
original_user_count = audition_df.distinct().count()

In [47]:
# Отображение полученных результатов

print(f"Пользователей в объединённой таблице: {joined_user_count}")
print(f"Пользователей в оригинальной таблице: {original_user_count}")

Пользователей в объединённой таблице: 1002896
Пользователей в оригинальной таблице: 1002896


* Используя `collect()`, выведем на экран все уникальные значения поля `main_content_type`.

In [48]:
# Все уникальные типы контента (используя collect)

unique_content_types = (
     new_df
    .select('main_content_type')
    .distinct()
    .collect()
)

In [49]:
# Вывод результатов

print("Уникальные main_content_type:")
for row in unique_content_types:
    print(row["main_content_type"])

Уникальные main_content_type:
Audiobook
None
Book
Comicbook


**ВЫВОДЫ ПО РАЗДЕЛУ 3**

1. Объеденены таблицы `audition_df` и `content_df` по столбцу `main_content_id`.

Было применено "левое" присоединение датасета `content_df` к датасету `audition_df`, потому что мы хотим сохранить все сессии из `audition_df`, даже если для какой-то сессии вдруг не найдётся совпадения в таблице контента (например, контент удалили, или ID потерялся). В таком случае поля из `content_df` будут `null`, но сама сессия не пропадёт. 

Таким образом, количество строк после объединения совпадает с количеством строк датасета `audition_df` (1002896).

2. Удалены все лишние столбцы из объединённой таблицы, которые не нужны для дальнейшего анализа: `main_author_id`, `app_version`, `usage_geo_id`.

3. Посчитано количество уникальных пользователей `puid` в объединённой таблице, которое совпадает с количеством строк в оригинальной таблице `audition_df` (1002896).

4. Было выявлено 3 типа уникального контента в сервисе: Audiobook, Book, Comicbook. Так же, в объединённом датасете присутствуют строки, где не указан тип контента (обусловлено "левым" присоединением).